# NB 1.2 &mdash; Solucions dels exercicis

**MP 5134** &mdash; UT1 · *Versió amb els pingüins de l'arxipèlag Palmer*

---

Solucionari dels sis exercicis de lectura de codi de la secció 6 del
[NB 1.2](NB_1_2_primer_model_PINGUINS.ipynb), i orientacions per al debat de la
secció 7.

Els quatre primers es responen **sense executar res**: són de lectura. Aquí,
a més, els comprovem, però a classe val la pena exigir la resposta escrita abans
de deixar tocar el teclat. Llegir codi i predir què farà és una habilitat que
s'entrena, i saltar-se-la és la manera més ràpida d'acabar programant a base de
provar coses fins que surti.

In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression

URL_DADES = "https://raw.githubusercontent.com/pprohenspolitecnicllevant/disseny-avaluacio-models-ml/refs/heads/main/UT01-Entorn_de_treball_primer_model/penguins/penguins.csv"
df = pd.read_csv(URL_DADES)

FEATURES = ["bill_length_mm", "bill_depth_mm", "flipper_length_mm"]
data = df[FEATURES + ["body_mass_g"]].dropna()

X = data[FEATURES]
y = data["body_mass_g"]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

model = LinearRegression().fit(X_train, y_train)
print(f"Punt de partida (el bloc de la secció 1): R2 = {model.score(X_test, y_test):.3f}")

## Exercici 1

> Quina línia fa que el model aprengui?

**Resposta:**

```python
model.fit(X_train, y_train)
```

Abans d'aquesta línia, `model` és un objecte buit: sap com és una regressió
lineal, però no sap res dels nostres pingüins. `fit()` és el mètode que li fa
recórrer les 273 mostres d'entrenament i calcular els coeficients.

Al NB 1.2 ho vam comprovar amb `hasattr(model, "coef_")`: abans de `fit()` dóna
`False`, després `True`.

*Per a la correcció:* l'error més freqüent és assenyalar `LinearRegression()`.
Val la pena aturar-s'hi: aquella línia **crea** el model, no l'entrena. És la
diferència entre comprar una llibreta i escriure-hi.

## Exercici 2

> Quina línia garanteix que l'avaluació sigui honesta?

**Resposta:**

```python
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
```

Aquesta línia aparta el 20 % de les dades **abans** d'entrenar. Gràcies a això,
quan després cridem `model.score(X_test, y_test)` estem preguntant sobre pingüins
que el model no ha vist mai.

En realitat l'honestedat la sostenen dues línies que treballen juntes: la partició
i el fet que `score()` rebi `X_test` i no `X_train`. Si algú respon les dues, ha
entès el mecanisme millor que qui en respon una.

**Comprovem-ho**, perquè la diferència es veu de seguida:

In [ ]:
# El mateix model, puntuat sobre les dades amb què ha entrenat i sobre les que no.
print(f"R2 sobre ENTRENAMENT (dades ja vistes): {model.score(X_train, y_train):.3f}")
print(f"R2 sobre TEST (dades noves):            {model.score(X_test, y_test):.3f}")

I aquí surt una sorpresa que va molt bé per a classe: **el test dóna més alt que
l'entrenament** (0,783 contra 0,756).

Molta gent espera el contrari, i amb raó a mitges. Un model que memoritza sí que
puntua més alt sobre les dades que ha vist &mdash; ho comprovaràs a l'exercici 5,
on un arbre sense podar treu 1,000 sobre l'entrenament i 0,606 sobre el test.
Però una regressió lineal amb tres variables **no té prou capacitat per
memoritzar res**: s'ajusta a la tendència general i punt. Quan no hi ha
memorització, quin dels dos números surt més alt depèn de quins 69 pingüins han
caigut al test, i aquesta vegada n'hi han caigut de fàcils.

La regla que sí que es manté sempre és l'altra: **la puntuació sobre
l'entrenament no serveix per estimar com de bé funcionarà el model amb dades
noves**. De vegades és massa optimista i de vegades no, i no saps quina de les
dues coses és sense un conjunt de prova.

*Per a la correcció:* si algú diu "la línia del `score()`", no és incorrecte, però
demana-li què passaria si el `score()` rebés `X_train`. Aquesta repregunta separa
qui ho ha entès de qui ho ha memoritzat.

## Exercici 3

> Què passaria si `FEATURES` inclogués `"body_mass_g"`? Quin valor donaria
> `score()` i per què això seria un error greu?

**Resposta esperada, per escrit:** donaria 1,0 o gairebé, i seria un error greu
perquè li estaríem donant al model la resposta com a entrada.

**Comprovem-ho:**

In [ ]:
# ATENCIÓ: aquesta cel·la fa a propòsit el que no s'ha de fer mai.
FEATURES_MALAMENT = ["bill_length_mm", "bill_depth_mm", "flipper_length_mm", "body_mass_g"]

# Compte: com que "body_mass_g" ja és dins de FEATURES_MALAMENT, no l'hem de
# tornar a afegir en seleccionar les columnes o tindríem la columna duplicada.
dades_fuga = df[FEATURES_MALAMENT].dropna()

X_fuga = dades_fuga[FEATURES_MALAMENT]     # la massa hi és com a ENTRADA...
y_fuga = dades_fuga["body_mass_g"]         # ...i alhora com a resposta

Xf_train, Xf_test, yf_train, yf_test = train_test_split(
    X_fuga, y_fuga, test_size=0.2, random_state=42
)

model_fuga = LinearRegression().fit(Xf_train, yf_train)
print(f"R2 amb la fuga: {model_fuga.score(Xf_test, yf_test):.6f}")

In [ ]:
# Per què surt exactament 1? Mirem què ha après el model.
for nom, coef in zip(FEATURES_MALAMENT, model_fuga.coef_):
    print(f"  {nom:>18} : {coef: .4f}")

**R2 = 1,000000.** Perfecte. I els coeficients expliquen per què: el de
`body_mass_g` val **1** i tots els altres valen **0**.

El model no ha après res de biologia. Ha descobert que una de les columnes
d'entrada *és* la resposta i s'ha limitat a copiar-la. La regla que ha après és
"la massa és igual a la massa".

Això és una **fuga d'informació** (*data leakage*), i el que la fa perillosa és que
**no dóna cap error**. El codi funciona, la puntuació és espectacular i el model és
completament inútil: el dia que li arribi un pingüí sense pesar, que és
precisament per a què el volíem, no tindrà la columna que necessita.

D'aquí ve el reflex que t'ha d'acompanyar tot el curs: **un resultat massa bo és
un motiu de sospita, no de celebració**. Aquí la fuga és òbvia perquè la columna
es diu igual. A la vida real s'amaga: una columna calculada a partir de la
resposta, una data posterior al fet que vols predir, un identificador que codifica
l'ordre en què es van registrar les coses. Hi tornarem a la **UT8** i a la
**UT10**.

## Exercici 4

> Si canvies `random_state=42` per `random_state=7`, canviarà el resultat? Molt o
> poc? Respon primer i comprova-ho després amb la taula de la secció 5.

In [ ]:
for llavor in [42, 7]:
    Xt, Xv, yt, yv = train_test_split(X, y, test_size=0.2, random_state=llavor)
    r2 = LinearRegression().fit(Xt, yt).score(Xv, yv)
    print(f"random_state={llavor:>2} -> R2 = {r2:.3f}")

**Resposta: sí, canvia, i més del que la majoria espera.** De **0,783** a
**0,750**: tres centèsimes i mitja.

El motiu és que `random_state` no canvia ni les dades ni el model: només decideix
**quins 69 pingüins van a parar al conjunt de prova**. Si en aquell grup hi cauen
pingüins fàcils de predir, el número puja; si hi cauen els rars, baixa.

I això passa perquè el conjunt és petit. A la taula de la secció 5 del NB 1.2, amb
cinc llavors, el R2 anava de 0,687 a 0,789: **una desena de diferència** sense
tocar res.

La conclusió pràctica és doble. Primera: **fixa sempre la llavor**, o no podràs
comparar dos models i saber si la diferència és mèrit del model o de la sort.
Segona, i més incòmoda: **si canviar la llavor mou el resultat més que canviar
el model, el teu experiment no demostra res**. Aquesta és la porta d'entrada a la
validació creuada de la **UT10**.

*Per a la correcció:* aquí el que s'avalua és la predicció escrita abans
d'executar. Qui digui "no canviarà, perquè les dades són les mateixes" ha comès
un error molt raonable i molt instructiu: val més equivocar-se aquí que a la UT11.

## Exercici 5

> Reescriu el bloc canviant `LinearRegression` per `DecisionTreeRegressor` (l'has
> d'importar de `sklearn.tree`). Quantes línies has hagut de tocar? Què et diu
> això sobre el disseny de scikit-learn?

In [ ]:
# Línia tocada 1: l'import.
from sklearn.tree import DecisionTreeRegressor

# Línia tocada 2: la classe que instanciem. Tota la resta és idèntica.
arbre = DecisionTreeRegressor(random_state=42)
arbre.fit(X_train, y_train)

print(f"Arbre de regressió:  R2 = {arbre.score(X_test, y_test):.3f}")
print(f"Regressió lineal:    R2 = {model.score(X_test, y_test):.3f}")

**Dues línies:** l'`import` i el nom de la classe. `fit()`, `predict()` i
`score()` es criden exactament igual.

Això és el que a scikit-learn anomenen **API consistent**, i és la raó per la qual
podrem comparar deu algorismes durant el curs sense reaprendre res cada vegada. Ho
vam anunciar a la secció 2.3 del NB 1.2 i aquí en tens la prova: canviar de
família d'algorisme costa dues línies.

**Però mira el número.** L'arbre treu **0,606** i la regressió lineal **0,783**.
L'arbre ho fa clarament pitjor.

In [ ]:
# Per què? Perquè l'arbre, sense límit de profunditat, memoritza l'entrenament.
print(f"Arbre sense límit  -> entrenament {arbre.score(X_train, y_train):.3f} | test {arbre.score(X_test, y_test):.3f}")

arbre_podat = DecisionTreeRegressor(max_depth=4, random_state=42).fit(X_train, y_train)
print(f"Arbre max_depth=4  -> entrenament {arbre_podat.score(X_train, y_train):.3f} | test {arbre_podat.score(X_test, y_test):.3f}")

Aquí hi ha regal per a la classe. L'arbre sense límit treu **1,000 sobre
l'entrenament** i 0,606 sobre el test: s'ha après els 273 pingüins de memòria,
un per un, i no ha entès cap patró general. Limitant-lo a profunditat 4 baixa a
0,864 sobre l'entrenament i puja a 0,718 sobre el test.

**Empitjorar el model sobre les dades d'entrenament l'ha fet millor sobre les
dades noves.** Sona contradictori i és una de les idees centrals del mòdul: es diu
**sobreajust** i té tota la UT2 i mitja UT5 dedicades.

I una lliçó de fons, que convé dir en veu alta: **canviar d'algorisme no és
sinònim de millorar**. Aquí el model més simple guanya. La massa d'un pingüí creix
de manera suau i contínua amb la mida de l'aleta, i això és exactament el que una
recta descriu bé i el que un arbre, que respon a escalons, descriu malament.

*Per a la correcció:* la resposta mínima és "dues línies, l'API és uniforme". La
resposta bona s'adona, a més, que el resultat ha empitjorat i es pregunta per què.

## Exercici 6

> Al NB 1.1 vam veure que els Gentoo són molt més pesants que les altres dues
> espècies. Si poguéssim afegir l'espècie com a característica, creus que el R2
> pujaria gaire? Deixa la resposta escrita: la comprovarem a la UT3, quan sapiguem
> convertir text en números.

L'exercici demanava només una predicció escrita, però com que el solucionari és
per a tu, aquí tens la comprovació. **A classe no la mostris encara**: perd tota
la gràcia com a obertura de la UT3.

In [ ]:
# get_dummies() és el one-hot de Pandas: converteix una columna de text en tantes
# columnes de 0/1 com valors diferents tingui. És el contingut de la UT3.
dades_sp = df[FEATURES + ["species", "body_mass_g"]].dropna()
X_sp = pd.get_dummies(dades_sp[FEATURES + ["species"]], columns=["species"])

print("Columnes després del one-hot:", list(X_sp.columns))
X_sp.head(3)

In [ ]:
Xs_train, Xs_test, ys_train, ys_test = train_test_split(
    X_sp, dades_sp["body_mass_g"], test_size=0.2, random_state=42
)
model_sp = LinearRegression().fit(Xs_train, ys_train)

print(f"Només amb les tres mesures:  R2 = {model.score(X_test, y_test):.3f}")
print(f"Afegint l'espècie:           R2 = {model_sp.score(Xs_test, ys_test):.3f}")

In [ ]:
# I si a més hi posem el sexe, que a l'exercici 6 del NB 1.1 vam veure que
# separava els mascles de les femelles dins de cada espècie:
dades_tot = df[FEATURES + ["species", "sex", "body_mass_g"]].dropna()
X_tot = pd.get_dummies(dades_tot[FEATURES + ["species", "sex"]], columns=["species", "sex"])

Xt_train, Xt_test, yt_train, yt_test = train_test_split(
    X_tot, dades_tot["body_mass_g"], test_size=0.2, random_state=42
)
print(f"Espècie + sexe:              R2 = {LinearRegression().fit(Xt_train, yt_train).score(Xt_test, yt_test):.3f}")

**Resposta: sí, i força.**

| Característiques | R2 |
|---|---|
| Les tres mesures | 0,783 |
| + espècie | **0,828** |
| + espècie + sexe | **0,873** |

Afegir l'espècie guanya quatre centèsimes i mitja; afegint també el sexe, nou en
total. I recorda que aquestes dues columnes **ja les teníem al fitxer des del
primer dia**: no hem sortit a mesurar cap pingüí més, només hem après a fer servir
informació que estàvem llençant.

Aquesta és exactament la promesa de la UT3, i és una bona manera d'obrir-la:
*preparar bé les variables sol donar més que canviar d'algorisme*. Compara-ho amb
l'exercici 5, on canviar de família d'algorisme ens va fer perdre disset
centèsimes.

*Per a la correcció:* qualsevol resposta raonada val, tant "sí perquè els Gentoo
són molt més pesants" com "no gaire, perquè l'aleta ja captura la mida i l'espècie
no hi afegeix res nou". La segona és més fina i és, de fet, mig certa: per això la
pujada és de quatre centèsimes i no de vint.

---

## Secció 7: orientacions per al debat

El debat no té solució, però sí que té respostes millors i pitjors. Això és el
que val la pena que surti:

**Un exemple ben plantejat** té les quatre peces sense confondre-les. La confusió
més habitual és entre la mostra i el conjunt de dades: *"la mostra és el
supermercat"* quan el que volien dir és *"cada venda del supermercat"*.

**Regressió o classificació** sol resoldre's bé, però apareix un cas fronterer
que val la pena aprofitar: predir una nota de 0 a 10. És regressió si la nota és
contínua, però *aprovat/suspès* és classificació. La mateixa realitat admet les
dues formulacions segons la decisió que hagis de prendre. És el mateix que fem
nosaltres amb els pingüins: la massa és regressió, l'espècie és classificació.

**El model de referència** és la pregunta nova d'aquesta sessió i la que costa
més. Respostes correctes: per a un preu, la mitjana dels preus; per a una
categoria, la categoria més freqüent; per a una sèrie temporal, *"demà igual que
avui"*. Si algú no sap dir quina és la seva referència, encara no ha entès prou
bé el seu propi problema.

**D'on sortirien les dades** és la pregunta que separa un projecte viable d'un
que no ho és. Aquí convé no acceptar *"d'internet"*. Les preguntes de tornada:
qui les recull, cada quant, què costa, i sobretot **existeix la resposta correcta
en aquestes dades?** Molts projectes moren aquí: hi ha les entrades però ningú no
ha registrat mai la sortida.

Un tancament que funciona bé: recordar que aquestes 344 files van costar **tres
campanyes a l'Antàrtida**. Les dades no apareixen, algú les fa.